Libraries

In [7]:
import os
import cv2
import numpy as np
import dtcwt
from tqdm import tqdm
from sklearn.preprocessing import normalize
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

DTCWT Transformer

In [8]:
transform = dtcwt.Transform2d()

Application of DTCWT and DNA-Encoding

In [9]:
def dtcwt_45_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray,(128,128))
    gray = gray.astype(np.float32)/255.0
    coeffs = transform.forward(gray,nlevels=3)
    feats = []

    for level in range(len(coeffs.highpasses)):
        subband = coeffs.highpasses[level][:,:,1]
        mag = np.abs(subband)
        feats.append(np.mean(mag))
        feats.append(np.std(mag))
        feats.append(np.sum(mag**2))

    return np.array(feats)

def dna_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    pixels = gray.flatten().astype(np.uint8)
    pair1 = (pixels >> 6) & 3
    pair2 = (pixels >> 4) & 3
    pair3 = (pixels >> 2) & 3
    pair4 = pixels & 3
    all_pairs = np.concatenate([pair1, pair2, pair3, pair4])
    counts = np.bincount(all_pairs, minlength=4)
    return counts / np.sum(counts)

Loading CNN features

In [10]:
f_cnn = np.load("/home/utkarshs/Desktop/Cbir_Video/Features/features_efficient.npy")
labels = np.load("/home/utkarshs/Desktop/Cbir_Video/Features/labels.npy")
paths = np.load("/home/utkarshs/Desktop/Cbir_Video/Features/paths.npy")


Extracting from paths and normalization

In [11]:
dtcwt_list = []
dna_list = []

for img_path in tqdm(paths):

    img = cv2.imread(img_path)

    if img is None:
        continue

    dtcwt_list.append(dtcwt_45_features(img))
    dna_list.append(dna_features(img))

f_dtcwt = np.array(dtcwt_list)
f_dna = np.array(dna_list)
print("DT-CWT shape:", f_dtcwt.shape)
print("DNA shape:", f_dna.shape)
f_cnn = normalize(f_cnn)
f_dtcwt = normalize(f_dtcwt)
f_dna = normalize(f_dna)

  0%|          | 0/9861 [00:00<?, ?it/s]/home/utkarshs/miniconda3/envs/Thesis/lib/python3.10/site-packages/dtcwt/utils.py:105: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(X, dtype=np.float32)
100%|██████████| 9861/9861 [00:26<00:00, 369.60it/s]

DT-CWT shape: (9861, 9)
DNA shape: (9861, 4)


Fusion

In [12]:
def fuse_features(w1, w2, w3):
    return np.concatenate([
        w1 * f_cnn,
        w2 * f_dtcwt,
        w3 * f_dna
    ], axis=1)

# Randomized weight search (weights are normalized to sum to 1)
rng = np.random.default_rng(42)
num_weight_samples = 200

rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_score = 0
best_weights = None

for _ in range(num_weight_samples):
    w1n, w2n, w3n = rng.dirichlet(np.ones(3))
    X = fuse_features(w1n, w2n, w3n)

    scores = cross_val_score(
        rf_model,
        X,
        labels,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    mean_score = scores.mean()

    if mean_score > best_score:
        best_score = mean_score
        best_weights = (w1n, w2n, w3n)
        print("\nNEW BEST")
        print("Weights:", best_weights)
        print("Accuracy:", best_score)

print("\n===== FINAL BEST RESULT =====")
print("Best weights:", best_weights)
print("Best accuracy:", best_score)

# Optional: fit final model on full data using best weights
X_best = fuse_features(*best_weights)
best_model = rf_model.fit(X_best, labels)
print("Best model:", best_model)

# Persist fused features using best weights
output_dir = "/home/utkarshs/Desktop/Cbir_Video/Database"
os.makedirs(output_dir, exist_ok=True)
np.save(os.path.join(output_dir, "fused_features_best.npy"), X_best)
np.save(os.path.join(output_dir, "fused_labels.npy"), labels)
np.save(os.path.join(output_dir, "fused_paths.npy"), paths)


NEW BEST
Weights: (np.float64(0.3374252443136454), np.float64(0.3278789386574905), np.float64(0.33469581702886425))
Accuracy: 0.4730740246882611

===== FINAL BEST RESULT =====
Best weights: (np.float64(0.3374252443136454), np.float64(0.3278789386574905), np.float64(0.33469581702886425))
Best accuracy: 0.4730740246882611
Best model: RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=42)
